# Lecture 7 Lab — From scikit-learn to PyTorch
### Machine Learning for Robotics & Industrial Automation

| | |
|---|---|
| **Estimated time** | 3–4 hours |
| **Tools** | Python, NumPy, pandas, matplotlib, scikit-learn, **PyTorch** |
| **Submit** | This completed notebook (see §10) |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# your name - surname, student ID


### Important cell below 

You must assign 3 last digits of your student ID to this <code>s_id</code> variable. For example, if your student ID is 6710546988, you must assign
```python
s_id = 988
```
This code will be printed in later cells. **Each output cell that shows incorrect s_id printout will get -1 penalty per cell.**

s_id may be used in some part of code as well, such as setting random seed. So the output for each student may be slightly different. 

In [ ]:
s_id = None  # replace with 3 last digits of your student ID.

## 1. Learning Objectives

By the end of this lab, you should be able to:

- Create PyTorch tensors and verify an autograd gradient against a hand calculation.
- Convert a scikit-learn preprocessing pipeline's output into tensors, a `Dataset`, and a `DataLoader`.
- Define a multilayer perceptron with `nn.Module` and train it with a hand-written training loop.
- Plot a loss curve and diagnose whether training worked.
- Compare a neural network against a classical baseline on identical data, and interpret the result honestly.

## 2. Setup

This week adds PyTorch. If you haven't installed it in your course environment yet:

```bash
pip install torch
```

(CPU-only is completely fine for this course — the models are small.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

torch.manual_seed(0)
np.random.seed(0)
print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available(), "(CPU is fine for this lab)")

## 3. Part A — Tensors & Autograd Warm-Up

Before trusting the framework on a real problem, verify it on one you can check by hand.

In [ ]:
# Tensors behave like NumPy arrays
# TODO: create the following tensors using torch
# a = [1.0, 2.0, 2.0]
# b = [0, 0, 0;
#      0, 0, 0]
a = None
b = None

print("Student ID : "+str(s_id))
print("a          :", a, "| shape", a.shape, "| dtype", a.dtype)
print("b.shape    :", b.shape)
print("a * 2      :", a * 2)
print("to numpy   :", a.numpy())
print("from numpy :", torch.from_numpy(np.array([4.0, 5.0])))

Now autograd. We'll compute the gradient of `y = x³ + 2x` at `x = 2`.

By hand: `dy/dx = 3x² + 2`, so at `x = 2` that's `3(4) + 2 = 14`.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3 + 2 * x
y.backward()
print("Student ID : "+str(s_id))
print(f"PyTorch says dy/dx = {x.grad.item()}")
print(f"By hand:      3x^2 + 2 = {3 * 2.0**2 + 2}")
assert abs(x.grad.item() - 14.0) < 1e-6, "Gradient does not match the hand calculation!"
print("\nMatch confirmed.")

### Your turn

Fill in the `TODO`: compute the gradient of `z = 5w² + 3w` at `w = 4` using autograd, and check it against your own derivative.

By hand, `dz/dw = 10w + 3`, so at `w = 4` it should be `43`.

In [ ]:
# TODO: compute the function z above at w=4
w = None
z = None

print("Student ID : "+str(s_id))
print(f"PyTorch says dz/dw = {w.grad.item()}   (expected 43.0)")

## 4. Part B — The Fault Dataset, Again

Same four-class fault dataset as Weeks 2 and 6 — deliberately. This week the *only* new thing is the framework, so nothing else should change.

In [ ]:
def generate_fault_dataset(n_per_class=150, seed=0):
    rng = np.random.default_rng(seed)
    class_names = ["Normal", "Bearing Fault", "Misalignment", "Imbalance"]
    centers = {
        "Normal":        dict(mean=0.02, std=0.15, rms=0.20, ptp=0.60),
        "Bearing Fault": dict(mean=0.03, std=0.35, rms=0.45, ptp=1.80),
        "Misalignment":  dict(mean=0.25, std=0.20, rms=0.55, ptp=0.90),
        "Imbalance":     dict(mean=0.05, std=0.18, rms=0.70, ptp=0.85),
    }
    rows = []
    added_var = 0.0001*s_id
    for label_idx, cname in enumerate(class_names):
        c = centers[cname]
        for _ in range(n_per_class):
            rows.append({
                "mean": rng.normal(c["mean"], 0.05+added_var),
                "std": rng.normal(c["std"], 0.04+added_var),
                "rms": rng.normal(c["rms"], 0.06+added_var),
                "ptp": rng.normal(c["ptp"], 0.15+added_var),
                "machine_type": rng.choice(["CNC", "Press", "Lathe"]),
                "label": label_idx,
            })
    df = pd.DataFrame(rows)
    for col in ["std", "rms"]:
        missing_idx = rng.choice(df.index, size=int(0.05 * len(df)), replace=False)
        df.loc[missing_idx, col] = np.nan
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


df = generate_fault_dataset(seed=s_id)
print("Student ID : "+str(s_id))
class_names = ["Normal", "Bearing Fault", "Misalignment", "Imbalance"]
numeric_features = ["mean", "std", "rms", "ptp"]
categorical_features = ["machine_type"]

X = df[numeric_features + categorical_features]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(f"Train: {len(X_train)}   Test: {len(X_test)}")

## 5. Part C — Preprocess, Then Convert to Tensors

Reuse the Week 2 `ColumnTransformer`. Neural networks need scaled inputs just as much as the SVM did — the pipeline handles that.

Note the dtypes: **features must be `float32`, class labels must be `int64`** (PyTorch's `CrossEntropyLoss` requires it). This is the single most common beginner error.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

# Fit on TRAIN only -- the Week 2 rule still applies
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Some transformers return a sparse matrix; make sure we have a dense array
X_train_proc = np.asarray(X_train_proc, dtype=np.float32)
X_test_proc = np.asarray(X_test_proc, dtype=np.float32)

X_train_t = torch.tensor(X_train_proc, dtype=torch.float32)
X_test_t = torch.tensor(X_test_proc, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.int64)
y_test_t = torch.tensor(y_test.values, dtype=torch.int64)
print("Student ID : "+str(s_id))
n_features = X_train_t.shape[1]
print(f"Feature tensor shape: {X_train_t.shape}  (4 numeric + 3 one-hot = {n_features} features)")
print(f"Label tensor dtype  : {y_train_t.dtype}")

In [ ]:
# TODO : Create tensor dataset and data loader 
train_ds = None
train_dl = None

print("Student ID : "+str(s_id))
# Peek at one batch to confirm shapes
xb, yb = next(iter(train_dl))
print("One batch -> features:", xb.shape, " labels:", yb.shape)

## 6. Part D — Define the Network

Fill in the `TODO`: build an MLP with two hidden layers (32 and 16 units), ReLU activations, and an output layer with one unit per class.

**Do not add a softmax at the end.** `nn.CrossEntropyLoss` applies it internally and expects raw logits — adding your own is the "softmax applied twice" bug from the lecture.

In [ ]:
class FaultNet(nn.Module):
    def __init__(self, n_features, n_classes=4):
        super().__init__()
        # TODO: build self.net as an nn.Sequential with:
        #   Linear(n_features -> 32), ReLU,
        #   Linear(32 -> 16), ReLU,
        #   Linear(16 -> n_classes)          <- no softmax!
        self.net = None

    def forward(self, x):
        return self.net(x)


model = FaultNet(n_features)
print("Student ID : "+str(s_id))
print(model)
print("\nTrainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7. Part E — Write the Training Loop

The four steps from the lecture, spelled out. Fill in the `TODO`s inside the inner loop.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

n_epochs = 100
epoch_losses = []
print("Student ID : "+str(s_id))
for epoch in range(n_epochs):
    model.train()
    batch_losses = []
    for xb, yb in train_dl:
        # TODO: the four steps, in order: (see slide 11)
        #   1. ?
        #   2. ?
        #   3. ?
        #   4. ?
        # Then append loss.item() to batch_losses.

    
    epoch_losses.append(np.mean(batch_losses))
    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch+1:3d}   mean loss: {epoch_losses[-1]:.4f}")

In [ ]:
print("Student ID : "+str(s_id))
plt.figure(figsize=(7, 4))
plt.plot(epoch_losses, color="#FF6A39", linewidth=2)
plt.xlabel("Epoch"); plt.ylabel("Mean cross-entropy loss")
plt.title("Training loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

> **What a healthy loss curve looks like:** a steep drop early, then flattening. If yours is flat from the start, the network isn't learning — check that you called `optimizer.step()`. If it's spiky and never settles, the learning rate is probably too high.

## 8. Part F — Evaluate the Network

`model.eval()` and `torch.no_grad()` turn off training-specific behavior and gradient tracking. Neither matters much for this small MLP, but both become essential in Week 9 once dropout and batch norm are in play — so build the habit now.

In [ ]:
model.eval()
with ? # TODO : fill in ? with correct code
    logits = model(X_test_t)
    nn_preds = logits.argmax(dim=1).numpy()

nn_acc = accuracy_score(y_test, nn_preds)
nn_f1 = f1_score(y_test, nn_preds, average="macro")
print("Student ID : "+str(s_id))
print(f"PyTorch MLP   accuracy: {nn_acc:.3f}   macro-F1: {nn_f1:.3f}")
print("\nConfusion matrix:")
print(pd.DataFrame(confusion_matrix(y_test, nn_preds), index=class_names, columns=class_names))

## 9. Part G — Compare Against the Classical Baseline

Same split, same preprocessing, same metric. Run a random forest and put the numbers side by side.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=0)
rf.fit(X_train_proc, y_train)
rf_preds = rf.predict(X_test_proc)

rf_acc = accuracy_score(y_test, rf_preds)
rf_f1 = f1_score(y_test, rf_preds, average="macro")

comparison = pd.DataFrame({
    "accuracy": [nn_acc, rf_acc],
    "macro_f1": [nn_f1, rf_f1],
}, index=["PyTorch MLP", "Random Forest"])
print(comparison.round(3))

comparison.plot(kind="bar", figsize=(6.5, 4), color=["#3E5C76", "#FF6A39"])
plt.ylabel("Score"); plt.ylim(0, 1.05); plt.xticks(rotation=0)
plt.title("Neural network vs. classical baseline — same data, same split")
plt.tight_layout()
plt.show()

> **Expect them to land close.** For small tabular datasets with hand-engineered features, that's the normal, honest result — and it's the reason the lecture opened by saying a tuned ensemble is often the better *engineering* choice here. The payoff for PyTorch arrives in Weeks 9–12, when the input becomes raw images, raw waveforms, and control policies that no tree can represent.

## 10. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below.

**1. In Part A, PyTorch's gradient matched your hand calculation exactly. Why does the framework need `requires_grad=True` — what would it have to store for *every* tensor if that flag didn't exist?**

*Your answer:* 

**2. Look at your loss curve from Part E. Did it flatten out? Would training for another 60 epochs likely help, and how can you tell from the curve?**

*Your answer:*

**3. How did the MLP compare to the random forest? Given the result, which would you actually deploy for this fault-detection task, and why?**

*Your answer:*

**4. We reused the Week 2 `ColumnTransformer` rather than feeding raw data to the network. Which preprocessing step was most important to keep for a neural network specifically, and what would likely have happened without it?**

*Your answer:* 


## 11. Deliverables & Submission

- This notebook, completed and able to run top-to-bottom without errors (`Kernel → Restart & Run All`).
- The loss curve from §7 and the comparison chart from §9.
- Your written answers to the four reflection questions.

Submit this `.ipynb` file in google classroom before the due date. Late penalty is -1 per day.

**Important :** If the notebook shows no output, or incorrect s_id, it will result in zero score. Check your work carefully before submitting. There is no dispute afterwards.

## 12. Grading Rubric Guide

| Component | Weight |
|---|---|
| Autograd exercise correct and verified by hand | 15% |
| Data correctly preprocessed and converted to tensors / DataLoader | 20% |
| `FaultNet` correctly defined (no softmax on the output) | 20% |
| Training loop correctly written; loss curve produced and sensible | 25% |
| Comparison against baseline + reflection questions | 10% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 10% |

Each output cell that shows incorrect s_id printout will get -1 penalty per cell.

---
**Next week:** *PyTorch Mechanics* — custom modules, optimizers, model saving, and a regression project predicting robot joint torque.

Generated by Claude and customized by

<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2026</div>